# Modelo 2 — AGCRN bimodal (SECA × PICO) · v3: losses equilibradas + agregação calibrada

Mesma arquitetura, features (SHAP corrigido), bancos `FINAL_NN_v2`, inferência alinhada e early stopping da v2.
Mudanças da v3 (marcadas `# [V3]`), motivadas pelo diagnóstico da v2 (`S1_corrigido/_analise_v2/`):

| Mudança | Por quê |
|---|---|
| **Losses novas** (`nb_loss_pico` e `nb_loss_seca`): NLL da Binomial Negativa pura + uma trava **simétrica** em escala log | As losses antigas eram assimétricas: a do PICO punia subestimar (e pesava mais as semanas de muitos casos) e a da SECA punia superestimar. Resultado na v2: o PICO previa **3–5× o real** no começo da temporada (SE 47–52) e ~1,8× no pico; a SECA previa ~0,65× |
| **Cada especialista também treina nas semanas do outro regime**, com peso `PESO_FORA_REGIME` | Antes cada um só via semanas do seu regime; o salto na troca SECA → PICO (SE 46 → 47) era de 3.600 para 17.000 casos previstos no Brasil com ~5.500 reais |
| **Pós-processamento calibrado na temporada de validação** (substitui o MinT) | O MinT calculava seus pesos com os erros do **próprio teste** (olhava a resposta). Agora: fator de viés por regime e UF, largura dos intervalos estaduais e soma estadual correta (distribuição da soma, não soma de medianas/quantis) — tudo ajustado só na validação |
| Checkpoints e resultados em pasta nova (`resultados_modelo2_v3`) | Não reaproveita os checkpoints da v2 (losses diferentes) |

Com os pesos da v2, só o pós-processamento novo já levou o erro estadual da S1 de MAE 829 (sem MinT) para **243**.

**Como usar:** os mesmos 3 arquivos da v2 em `MyDrive/dengue_S1/` (`train1_FINAL_NN_v2.parquet`, `test1_FINAL_NN_v2.parquet`,
`ctx1_FINAL_NN_v2.parquet`), GPU T4 e *Executar tudo*. Para só refazer a previsão com pesos já treinados, use `SO_POS_PROCESSAMENTO = True`.

## 1. Pacotes

In [ ]:
# @title Conferir ambiente (não é preciso instalar nada)
# A versão anterior instalava torch 2.4 + torch-geometric-temporal. O Colab atual não tem mais o torch 2.4
# e tentava COMPILAR torch-scatter/torch-sparse (dezenas de minutos). Agora a AGCRN está embutida
# no notebook (célula 1b) e usa o PyTorch que já vem no Colab.
import torch, numpy, pandas, scipy, pyarrow, sys
print('Python', sys.version.split()[0], '| torch', torch.__version__, '| numpy', numpy.__version__,
      '| pandas', pandas.__version__, '| pyarrow', pyarrow.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NENHUMA — ative a GPU T4')

In [ ]:
# @title Importar pacotes
import os, sys, gc, time, random, sqlite3, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import nbinom, pearsonr
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore', category=FutureWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Hardware: {device.type.upper()}", torch.cuda.get_device_name(0) if device.type == 'cuda' else '')
if device.type != 'cuda':
    print("⚠️ SEM GPU: o treino levaria dias. Ative a GPU em 'Ambiente de execução → Alterar tipo'.")

In [ ]:
# @title AGCRN (mesma implementação de torch_geometric_temporal.nn.recurrent.AGCRN, só PyTorch)
# Embutida aqui para não depender de torch-geometric-temporal / torch-scatter / torch-sparse,
# que exigem versões antigas do torch e precisam ser compiladas no Colab atual.
import math


def _glorot(t):
    a = math.sqrt(6.0 / (t.size(-2) + t.size(-1)))
    with torch.no_grad():
        t.uniform_(-a, a)


class AVWGCN(nn.Module):
    """Adaptive Graph Convolution com pesos por nó (Bai et al., 2020)."""
    def __init__(self, in_channels: int, out_channels: int, K: int, embedding_dimensions: int):
        super().__init__()
        self.K = K
        self.weights_pool = nn.Parameter(torch.empty(embedding_dimensions, K, in_channels, out_channels))
        self.bias_pool = nn.Parameter(torch.empty(embedding_dimensions, out_channels))
        _glorot(self.weights_pool)
        nn.init.zeros_(self.bias_pool)

    def forward(self, X: torch.Tensor, E: torch.Tensor) -> torch.Tensor:
        number_of_nodes = E.shape[0]
        supports = F.softmax(F.relu(torch.mm(E, E.transpose(0, 1))), dim=1)
        support_set = [torch.eye(number_of_nodes, device=supports.device, dtype=supports.dtype), supports]
        for _ in range(2, self.K):
            support_set.append(torch.matmul(2 * supports, support_set[-1]) - support_set[-2])
        supports = torch.stack(support_set, dim=0)
        W = torch.einsum('nd,dkio->nkio', E, self.weights_pool)
        bias = torch.matmul(E, self.bias_pool)
        X_G = torch.einsum('knm,bmc->bknc', supports, X)
        X_G = X_G.permute(0, 2, 1, 3)
        return torch.einsum('bnki,nkio->bno', X_G, W) + bias


class AGCRN(nn.Module):
    """Célula recorrente AGCRN (GRU com convolução de grafo adaptativa)."""
    def __init__(self, number_of_nodes: int, in_channels: int, out_channels: int, K: int, embedding_dimensions: int):
        super().__init__()
        assert K > 1, 'K deve ser > 1'
        self.number_of_nodes = number_of_nodes
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.K = K
        self.embedding_dimensions = embedding_dimensions
        self._gate = AVWGCN(in_channels + out_channels, 2 * out_channels, K, embedding_dimensions)
        self._update = AVWGCN(in_channels + out_channels, out_channels, K, embedding_dimensions)

    def forward(self, X: torch.Tensor, E: torch.Tensor, H: torch.Tensor = None) -> torch.Tensor:
        if H is None:
            H = torch.zeros(X.shape[0], X.shape[1], self.out_channels, device=X.device, dtype=X.dtype)
        X_H = torch.cat((X, H), dim=-1)
        Z_R = torch.sigmoid(self._gate(X_H, E))
        Z, R = torch.split(Z_R, self.out_channels, dim=-1)
        C = torch.cat((X, Z * H), dim=-1)
        HC = torch.tanh(self._update(C, E))
        return R * H + (1 - R) * HC

## 2. Configuração, caminhos e variáveis

In [ ]:
# @title Configuração  # [NOVO]
SEASON_ALVO = 1
SEED = 42                      # fixa a inicialização da rede (use o mesmo valor para comparar execuções)
RETOMAR = True                 # True: se existir checkpoint desta season, continua dele. False: treina do zero
CKPT_A_CADA_EPOCAS = 1         # salva checkpoint local a cada N épocas (cópia no Drive a cada 5)
EPOCHS_MAX = 200               # teto; o early stopping costuma parar bem antes
# [V2] EARLY STOPPING (monitora a loss de validação de cada especialista, no próprio regime)
PATIENCE = 15                  # épocas seguidas sem melhora até parar
MIN_DELTA = 1e-3               # melhora mínima de 0,1% na loss para contar como melhora
EPOCHS_MIN = 10                # nunca para antes disso
SEMANAS_VALIDACAO = 52         # validação = última temporada do treino (EW26 -> EW25): tem semanas de SECA e de PICO
# [V3] LOSS E TREINO
LAMBDA_TRAVA_LOG = 0.2         # peso da trava simétrica em escala log (0 = só a NLL da Binomial Negativa)
PESO_FORA_REGIME = 0.25        # cada especialista também aprende as semanas do outro regime com esse peso (0 = como antes)
# [V3] PÓS-PROCESSAMENTO (tudo ajustado só na temporada de validação)
ENCOLHIMENTO_UF = 2000         # fator de viés por UF, encolhido para o nacional (0 = um fator nacional por regime)
SO_POS_PROCESSAMENTO = False   # True: não treina; usa os pesos modelo_agcrn_*_s{N}.pt já salvos e refaz só a previsão
USAR_TORCH_COMPILE = True      # se der erro mencionando 'inductor'/'dynamo'/'compile', mude para False e rode de novo

if not USAR_TORCH_COMPILE:
    torch.compile = lambda modelo, *a, **k: modelo   # roda sem compilação (um pouco mais lento, mesmo resultado)

try:
    import google.colab  # noqa
    EM_COLAB = True
except ImportError:
    EM_COLAB = False
EM_KAGGLE = os.path.exists('/kaggle/working')

if EM_KAGGLE:
    PASTA_DADOS = f'/kaggle/input/dengue-s{SEASON_ALVO}'   # nome do Dataset que você criar no Kaggle
    PASTA_SAIDA = '/kaggle/working'
elif EM_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    PASTA_DADOS = f'/content/drive/MyDrive/dengue_S{SEASON_ALVO}'                            # onde estão os 3 parquets v2
    PASTA_SAIDA = f'/content/drive/MyDrive/dengue_S{SEASON_ALVO}/resultados_modelo2_v3'  # [V3] pasta nova
else:
    PASTA_DADOS = '.'
    PASTA_SAIDA = './resultados_modelo2_v3'

# [NOVO] Trabalhar no DISCO LOCAL da sessão. O Drive só é usado para (1) ler os dados uma vez e
# (2) guardar cópias de checkpoints/resultados. Se o Drive cair, o treino continua.
import shutil
PASTA_DADOS, PASTA_SAIDA = os.path.abspath(PASTA_DADOS), os.path.abspath(PASTA_SAIDA)
PASTA_LOCAL = '/content/trabalho_v3' if EM_COLAB else PASTA_SAIDA
os.makedirs(PASTA_LOCAL, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)
for f in (f"train{SEASON_ALVO}_FINAL_NN_v2.parquet", f"test{SEASON_ALVO}_FINAL_NN_v2.parquet",   # [V2]
          f"ctx{SEASON_ALVO}_FINAL_NN_v2.parquet"):
    origem, destino = os.path.join(PASTA_DADOS, f), os.path.join(PASTA_LOCAL, f)
    assert os.path.exists(origem) or os.path.exists(destino), f"Não encontrei {f} em {PASTA_DADOS}"
    if not os.path.exists(destino):
        print(f"📥 Copiando {f} do Drive para o disco local ...")
        shutil.copy(origem, destino)
# traz checkpoints/banco já existentes no Drive (para RETOMAR depois de uma queda)
for f in os.listdir(PASTA_SAIDA):
    if (f.startswith('ckpt_') or f.startswith('modelo_agcrn_') or f == 'resultados_agcrn.db') and not os.path.exists(os.path.join(PASTA_LOCAL, f)):
        shutil.copy(os.path.join(PASTA_SAIDA, f), os.path.join(PASTA_LOCAL, f))
PASTA_DADOS = PASTA_LOCAL
os.chdir(PASTA_LOCAL)
DB_LOCAL = os.path.join(PASTA_LOCAL, 'resultados_agcrn.db')


def salvar_no_drive(arquivo):
    """Copia um arquivo local para o Drive. Se o Drive estiver fora do ar, avisa e segue."""
    if PASTA_LOCAL == PASTA_SAIDA:
        return
    try:
        shutil.copy(arquivo, os.path.join(PASTA_SAIDA, os.path.basename(arquivo)))
    except OSError as e:
        print(f"   ⚠️ não consegui copiar {arquivo} para o Drive ({e}); o treino continua")


print('📂 Trabalhando em:', os.getcwd())
print('💾 Cópias no Drive:', PASTA_SAIDA)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
# @title Variáveis do banco de dados (célula 3 do notebook original)
VARS = {
    'municipio':    'geocode',
    'semana_epi':   'epiweek',
    'data':         'date',
    'casos_dengue': 'casos.x',
    'populacao':    'population',
    'uf_code':      'uf_code',
}
SEASONS = {
    1: {'train': "train1_dengue.parquet", 'test': "test1_dengue.parquet", 'label': '2022-2023'},
    2: {'train': "train2_dengue.parquet", 'test': "test2_dengue.parquet", 'label': '2023-2024'},
    3: {'train': "train3_dengue.parquet", 'test': "test3_dengue.parquet", 'label': '2024-2025'},
    4: {'train': "train4_dengue.parquet", 'test': "test4_dengue.parquet", 'label': '2025-2026'},
}
QUANTIS = {'50': (0.25, 0.75), '80': (0.10, 0.90), '90': (0.05, 0.95), '95': (0.025, 0.975)}

## 3. Pós-processamento: calibração na validação + soma estadual (substitui o MinT)

In [ ]:
# @title [V3] Previsão + calibração na validação + soma estadual (substitui o MinT)
# Por que não o MinT: ele calculava os pesos com os erros do PRÓPRIO TESTE (casos_reais), isto é, olhava a
# resposta; e, como o estado era a soma das medianas municipais, a entrada já era coerente e o MinT só
# encolhia as maiores cidades por um efeito da regularização.
#
# O que esta célula faz, usando só a TEMPORADA DE VALIDAÇÃO (últimas SEMANAS_VALIDACAO semanas do treino):
#  1. FATOR DE VIÉS por regime (PICO/SECA) e por UF = casos reais / casos previstos na validação,
#     encolhido para o fator nacional quando a UF tem pouco volume (ENCOLHIMENTO_UF).
#  2. SOMA ESTADUAL: média = soma das médias municipais; variância = soma das variâncias × k, representada
#     por uma Binomial Negativa. k (por regime) corrige a correlação entre municípios vizinhos e é escolhido
#     para que os IC50/IC90 estaduais da validação fiquem perto do nominal.
#     O PONTO (pred) do estado é a mediana SEM a inflação; a inflação só alarga os intervalos (que sempre contêm o ponto).
#  3. Previsão alinhada: a semana t-1 prevê a semana t; o estado da rede é aquecido nas 15 semanas anteriores.
import pyarrow.parquet as pq
from scipy.stats import nbinom

AQUEC_PREV = 15
GRADE_K = [1, 1.5, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64]
QUANTIS_POS = {'lower_95': .025, 'lower_90': .05, 'lower_80': .10, 'lower_50': .25, 'pred': .50,
           'upper_50': .75, 'upper_80': .90, 'upper_90': .95, 'upper_95': .975}


def _nb_estadual(mu, r, A, k=1.0):
    """Quantis da soma por UF: NB com média = soma das médias e variância = soma das variâncias × k."""
    m = mu @ A.T
    v = (mu + mu ** 2 / r) @ A.T
    rr = np.where(v > m, m ** 2 / np.maximum(k * (v - m), 1e-9), 1e6)
    return {c: nbinom.ppf(q, rr, rr / (rr + m)) for c, q in QUANTIS_POS.items()}


def prever_e_agregar_calibrado(season_id: int):
    t0 = time.time()
    torch.set_grad_enabled(False)
    # ------------------------------------------------------------ dados (ordem date, node_id)
    arq_tr = os.path.join(PASTA_DADOS, f"train{season_id}_FINAL_NN_v2.parquet")
    eps = np.sort(pq.read_table(arq_tr, columns=['epiweek'])['epiweek'].to_pandas().unique())
    df_val = pd.read_parquet(arq_tr, filters=[('epiweek', '>=', int(eps[-(SEMANAS_VALIDACAO + AQUEC_PREV)]))])
    df_ctx = pd.read_parquet(os.path.join(PASTA_DADOS, f"ctx{season_id}_FINAL_NN_v2.parquet"))
    df_te = pd.read_parquet(os.path.join(PASTA_DADOS, f"test{season_id}_FINAL_NN_v2.parquet"))
    df_val, df_ctx, df_te = [d[d[VARS['uf_code']] != 32].sort_values(['date', 'node_id']).reset_index(drop=True)
                             for d in (df_val, df_ctx, df_te)]
    N = df_te['node_id'].nunique()
    uf_no = df_te[VARS['uf_code']].to_numpy()[:N]
    ufs = np.unique(uf_no)
    A = (uf_no[None, :] == ufs[:, None]).astype(float)              # (UF, município)

    # ------------------------------------------------------------ modelos
    def carregar(nome, cols):
        m = PreditorNBAGCRN(N, len(cols), 64, cols.index('velocidade'), cols.index('aceleracao')).to(device)
        arq = f"modelo_agcrn_{nome}_s{season_id}.pt"
        caminho = arq if os.path.exists(arq) else os.path.join(PASTA_SAIDA, arq)
        sd = torch.load(caminho, map_location=device, weights_only=False)
        m.load_state_dict({k.replace('_orig_mod.', ''): v.float() for k, v in sd.items()})
        print(f"📦 Pesos: {caminho}")
        return m.eval()
    mod_s, mod_p = carregar('SECA', COLS_SECA), carregar('PICO', COLS_PICO)

    def tens(df, cols):
        n = df['date'].nunique()
        return torch.tensor(df[cols].to_numpy(np.float32).reshape(n, N, len(cols)), device=device)

    def prever(df_aq, df_alvo):
        df = pd.concat([df_aq, df_alvo], ignore_index=True)
        Xs, Xp = tens(df, COLS_SECA), tens(df, COLS_PICO)
        lp = torch.tensor(df['log_pop'].to_numpy(np.float32).reshape(-1, N), device=device)
        n_aq, n_alvo = df_aq['date'].nunique(), df_alvo['date'].nunique()
        se = df_alvo.groupby('date')[VARS['semana_epi']].first().to_numpy() % 100
        pico = (se >= 47) | (se <= 25)
        mu, r = np.zeros((n_alvo, N)), np.zeros((n_alvo, N))
        hs = hp = None
        for t in range(n_aq + n_alvo - 1):
            lms, lrs, hs = mod_s(Xs[t], lp[t], hs)
            lmp, lrp, hp = mod_p(Xp[t], lp[t], hp)
            k = t + 1 - n_aq
            if k < 0:
                continue
            mu[k] = torch.exp(lmp if pico[k] else lms).squeeze(-1).float().cpu().numpy()
            r[k] = torch.exp(lrp if pico[k] else lrs).squeeze(-1).float().cpu().numpy()
        return np.maximum(mu, 1e-8), np.maximum(r, 1e-3), pico

    # ------------------------------------------------------------ calibração na validação
    sem_val = df_val[VARS['semana_epi']].unique()
    alvo_val = df_val[df_val[VARS['semana_epi']].isin(sem_val[AQUEC_PREV:])]
    mu_v, r_v, pico_v = prever(df_val[df_val[VARS['semana_epi']].isin(sem_val[:AQUEC_PREV])], alvo_val)
    y_v = alvo_val[VARS['casos_dengue']].to_numpy(np.float64).reshape(-1, N)
    yv_uf = y_v @ A.T
    fator_no, K = {}, {}
    print(f"🎯 Calibração na validação (EW {alvo_val[VARS['semana_epi']].min()}–{alvo_val[VARS['semana_epi']].max()}):")
    for nome, m in (('pico', pico_v), ('seca', ~pico_v)):
        g = y_v[m].sum() / mu_v[m].sum()
        num, den = yv_uf[m].sum(0), (mu_v[m] @ A.T).sum(0)
        w = den / (den + ENCOLHIMENTO_UF) if ENCOLHIMENTO_UF > 0 else np.zeros_like(den)
        f_uf = np.where(den > 0, w * num / np.maximum(den, 1e-9) + (1 - w) * g, g)
        fator_no[nome] = f_uf @ A
        mu_c = mu_v[m] * fator_no[nome]
        erros = []
        for k in GRADE_K:
            q = _nb_estadual(mu_c, r_v[m], A, k)
            c50 = np.mean((yv_uf[m] >= q['lower_50']) & (yv_uf[m] <= q['upper_50'])) * 100
            c90 = np.mean((yv_uf[m] >= q['lower_90']) & (yv_uf[m] <= q['upper_90'])) * 100
            erros.append(abs(c50 - 50) + abs(c90 - 90))
        K[nome] = GRADE_K[int(np.argmin(erros))]
        print(f"   {nome.upper()}: fator nacional × {g:.3f} · por UF de × {f_uf.min():.2f} a × {f_uf.max():.2f} · "
              f"inflação da variância estadual k = {K[nome]}")

    # ------------------------------------------------------------ teste
    mu, r, pico_t = prever(df_ctx, df_te)
    mu = mu * np.where(pico_t[:, None], fator_no['pico'][None, :], fator_no['seca'][None, :])
    p = r / (r + mu)
    df_mun = df_te[[VARS['municipio'], 'date', VARS['uf_code'], VARS['casos_dengue']]] \
        .rename(columns={VARS['casos_dengue']: 'casos_reais'}).reset_index(drop=True)
    for col, q in QUANTIS_POS.items():
        df_mun[col] = nbinom.ppf(q, r, p).ravel().round(2)
    datas_te = sorted(df_te['date'].unique())
    q_est = {c: np.zeros((len(datas_te), len(ufs))) for c in QUANTIS_POS}
    for nome, m in (('pico', pico_t), ('seca', ~pico_t)):
        # [V3.1] ponto = mediana da soma SEM inflação. Com k grande (ex.: SECA k=48 na S1), a mediana da NB inflada
        # despenca para perto de zero (a distribuição fica muito assimétrica) e a previsão estadual colapsava
        # no início e no fim da temporada, embora a média estivesse certa.
        # Intervalos = união do intervalo sem inflação com o inflado: nunca mais estreitos que o calibrado e
        # sempre contendo o ponto.
        q = _nb_estadual(mu[m], r[m], A, K[nome])
        q1 = _nb_estadual(mu[m], r[m], A, 1.0)
        for c in QUANTIS_POS:
            if c.startswith('lower'):
                q_est[c][m] = np.minimum(q[c], q1[c])
            elif c.startswith('upper'):
                q_est[c][m] = np.maximum(q[c], q1[c])
            else:
                q_est[c][m] = q1[c]
    df_est = pd.DataFrame({VARS['uf_code']: np.tile(ufs, len(datas_te)), 'date': np.repeat(datas_te, len(ufs))})
    for c in QUANTIS_POS:
        df_est[c] = q_est[c].ravel().round(2)
    real = df_te.groupby([VARS['uf_code'], 'date'])[VARS['casos_dengue']].sum().rename('casos_reais').reset_index()
    df_est = df_est.merge(real, on=[VARS['uf_code'], 'date'], how='left')
    torch.set_grad_enabled(True)
    print(f"✅ Previsão calibrada pronta em {time.time() - t0:.0f}s")
    return df_mun, df_est


## 4. Modelo 2: funções de perda, features e arquitetura (células 41–45)

In [ ]:
# @title [V3] NB loss pico e seca (equilibradas)
import torch
import torch.nn.functional as F
import numpy as np
from scipy.stats import nbinom


def _nb_nll(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor):
    """NLL da Binomial Negativa (média mu, dispersão r), em float32 mesmo sob autocast."""
    log_mu = torch.clamp(log_mu.float(), min=-10, max=12)
    log_r = torch.clamp(log_r.float(), min=-6, max=6)
    y = y.float()
    r = torch.exp(log_r)
    log_r_mais_mu = torch.logaddexp(log_r, log_mu)                  # log(r + mu), estável
    nll = (torch.lgamma(r) + torch.lgamma(y + 1) - torch.lgamma(y + r)
           - r * (log_r - log_r_mais_mu) - y * (log_mu - log_r_mais_mu))
    return nll, log_mu


def nb_loss_equilibrada(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor,
                        lam: float = None) -> torch.Tensor:
    """NLL da NB (regra de pontuação própria: sem viés para cima nem para baixo)
    + trava SIMÉTRICA contra erros grosseiros na escala log: prever 3x a mais custa o mesmo que prever 3x a menos,
    com a mesma regra para semanas de poucos e de muitos casos."""
    lam = LAMBDA_TRAVA_LOG if lam is None else lam
    nll, log_mu_c = _nb_nll(log_mu, log_r, y)
    erro_log = torch.log1p(torch.exp(log_mu_c)) - torch.log1p(y.float())
    trava = F.smooth_l1_loss(erro_log, torch.zeros_like(erro_log), beta=1.0, reduction='none')
    return (nll + lam * trava).mean()


# Os dois especialistas usam a MESMA loss: o que os diferencia são as semanas em que cada um tem peso
# cheio no treino (PESO_FORA_REGIME) e as semanas em que cada um é usado na previsão.
def nb_loss_pico(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return nb_loss_equilibrada(log_mu, log_r, y)

In [ ]:
# @title [V3] NB loss seca
def nb_loss_seca(log_mu: torch.Tensor, log_r: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    return nb_loss_equilibrada(log_mu, log_r, y)

In [ ]:
# @title Criar Quantis
def nb_quantile(mu: np.ndarray, r: np.ndarray, q: float) -> np.ndarray:
    p = r / (r + mu + 1e-8)
    return nbinom.ppf(q, n=r, p=p).astype(float)

In [ ]:
# @title LISTAS DE FEATURES PARA O AGCRN (Atualizadas com nomes pós-transformação)


# Variáveis físicas básicas que o modelo precisa saber a posição exata
# [V2] listas do SHAP corrigido (SHAP_corrigido/listas_features_modelo2.py)
# Gerado por SHAP_corrigido/shap_walkforward_corrigido.py em 2026-09-29 12:36 (seasons [1, 2, 3, 4])
# Critério: Top 25 por season e regime, acima das sondas de ruído na permutação (>= 2 de 3 folds), sem redundância (|r| > 0.95), presente em >= 3 de 4 seasons. Ordem: importância SHAP média.
# Requer train{N}_FINAL_NN_v2.parquet se usar alguma destas colunas novas: ['log_inc_dengue_obs', 'log_inc_dengue_obs_lag1', 'log_inc_dengue_obs_lag2', 'w1_log_inc_dengue_obs', 'w2_log_inc_dengue_obs', 'temp_min_lag2', 'w1_pressure_min_t1', 'precip_max_lag3', 'biome_encoded']

COLS_FISICA = ['velocidade', 'aceleracao']

# 🌵 SECA (20 features)
COLS_SECA = COLS_FISICA + [
    'log_inc_dengue_t1', 'log_inc_dengue_obs', 'log_inc_dengue_obs_lag1', 'imunidade_ewm',
    'log_pop_t1', 'log_inc_dengue_obs_lag2', 'w1_log_inc_dengue_obs', 'w2_log_inc_dengue_obs',
    'w1_log_inc_dengue_t1', 'w2_temp_forecast_t1', 'w2_log_inc_dengue_t1', 'cosseno_semana',
    'w1_log_pop_t1', 'precip_tot_forecast', 'rel_humid_max', 'temp_min_lag2',
    'w1_pressure_min_t1', 'temp_min_lag4',
]

# 🌊 PICO (20 features)
COLS_PICO = COLS_FISICA + [
    'log_inc_dengue_obs', 'log_inc_dengue_t1', 'log_inc_dengue_obs_lag1', 'w1_log_inc_dengue_obs',
    'w2_log_inc_dengue_obs', 'log_inc_dengue_obs_lag2', 'cosseno_semana', 'log_pop_t1',
    'imunidade_ewm', 'temp_med', 'w1_log_pop_t1', 'seno_semana',
    'w1_velocidade_t1', 'w2_velocidade_t1', 'w2_log_inc_dengue_t1', 'precip_max_lag3',
    'biome_encoded', 'koppen_encoded',
]


In [ ]:
# @title NOVO AGCRN padrão
import torch
import torch.nn as nn
import torch.nn.functional as F

class PreditorNBAGCRN(nn.Module):
    def __init__(self, num_nodes: int, num_features: int, embedding_dim: int = 64, idx_velocidade: int = 0, idx_aceleracao: int = 1):
        super().__init__()

        # Recebe dinamicamente os índices da velocidade e aceleração
        self.idx_vel = idx_velocidade
        self.idx_acel = idx_aceleracao

        # Node Embeddings (Aprendem o viés estrutural de cada cidade)
        self.node_embeddings = nn.Parameter(torch.randn(num_nodes, embedding_dim), requires_grad=True)

        self.recurrent = AGCRN(
            number_of_nodes=num_nodes,
            in_channels=num_features,
            out_channels=64,
            K=2,                                   # Ordem de vizinhança espacial
            embedding_dimensions=embedding_dim     # Passando a dimensão do embedding
        )

        # 🔥 ALTERAÇÃO: Inclusão de Dropout Espaço-Temporal
        self.dropout = nn.Dropout(p=0.2)

        self.head_mu = nn.Linear(64, 1)
        # Head do parâmetro R recebe o hidden state + as 2 variáveis físicas brutas (vel e acel)
        self.head_logr = nn.Linear(64 + 2, 1)

    def forward(self, x: torch.Tensor, log_pop: torch.Tensor, h=None):
        # 1. Adiciona a dimensão de Batch
        x_batched = x.unsqueeze(0)

        # 2. Passa pela rede recorrente (Geração do estado oculto espaço-temporal)
        h = self.recurrent(x_batched, self.node_embeddings, h)

        h_out = h.squeeze(0)
        h_relu = F.relu(h_out)

        # 🔥 CORREÇÃO: O dropout agora é devidamente aplicado antes de calcular mu e r
        h_reg = self.dropout(h_relu)

        # Cálculo do log_mu utilizando o tensor com Dropout (h_reg)
        log_mu_raw = self.head_mu(h_reg)
        log_mu = log_mu_raw + log_pop.unsqueeze(-1)

        # Resgate da física diretamente da matriz de entrada usando os índices dinâmicos
        vel = x[:, self.idx_vel].unsqueeze(-1)
        acel = x[:, self.idx_acel].unsqueeze(-1)

        # Concatena o estado oculto regularizado (h_reg) com a física
        h_r = torch.cat([h_reg, vel, acel], dim=-1)
        log_r = self.head_logr(h_r)

        return log_mu, log_r, h

## 5. Treino e validação bifurcados (célula 46)

In [ ]:
# @title NOVO TREINO E VALIDAÇÃO
# ==============================================================================
# FUNÇÃO PRINCIPAL: TREINO E VALIDAÇÃO BIFURCADOS
# ==============================================================================
def treinar_e_validar_season(df_train: pd.DataFrame, df_test: pd.DataFrame,
                             season_id: int, encoder_means_globais: dict,
                             num_nodes: int, mapa_simples: pd.DataFrame,
                             epochs_max: int = 150, patience: int = 25) -> dict:

    torch.backends.cudnn.benchmark = True

    # 🔥 OTIMIZAÇÃO DE RAM 2: Fim do .copy(). Usaremos os ponteiros de memória originais!
    df_tr, df_tgt = df_train, df_test

    if 'log_pop' not in df_tr.columns:
        df_tr['log_pop'] = np.log(df_tr[VARS['populacao']].clip(lower=1).astype(float))
    if 'log_pop' not in df_tgt.columns:
        df_tgt['log_pop'] = np.log(df_tgt[VARS['populacao']].clip(lower=1).astype(float))

    # [V2] validação = última temporada completa do treino (SEMANAS_VALIDACAO semanas), para que os DOIS
    # especialistas tenham semanas de validação. Antes eram as 10 últimas (EW16–25, todas de pico): a SECA
    # nunca validava e rodava até EPOCHS_MAX. As AQUEC_VAL semanas anteriores só aquecem o estado recorrente.
    AQUEC_VAL = 15
    semanas_treino = sorted(df_tr[VARS['semana_epi']].unique())
    df_tr_real = df_tr[df_tr[VARS['semana_epi']].isin(semanas_treino[:-SEMANAS_VALIDACAO])]
    df_val_int = df_tr[df_tr[VARS['semana_epi']].isin(semanas_treino[-(SEMANAS_VALIDACAO + AQUEC_VAL):])]
    _sv = np.array(semanas_treino[-SEMANAS_VALIDACAO:]) % 100
    _n_pico = int(((_sv >= 47) | (_sv <= 25)).sum())
    print(f"📏 Treino: {len(semanas_treino) - SEMANAS_VALIDACAO} semanas ({semanas_treino[0]}–{semanas_treino[-SEMANAS_VALIDACAO - 1]}) · "
          f"Validação: {SEMANAS_VALIDACAO} semanas ({semanas_treino[-SEMANAS_VALIDACAO]}–{semanas_treino[-1]}): "
          f"{SEMANAS_VALIDACAO - _n_pico} de SECA e {_n_pico} de PICO")

    # --------------------------------------------------------------------------
    # 1. PURGA ABSOLUTA DOS ÍNDICES BUGADOS
    # Removemos node_id antigo e time_index. Ordenamos pela 'data' (inquebrável)
    # --------------------------------------------------------------------------
    df_tr_sort  = df_tr_real.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])
    df_val_sort = df_val_int.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])
    df_tgt_sort = df_tgt.drop(columns=['node_id', 'time_index'], errors='ignore').merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])

    # --------------------------------------------------------------------------
    # 2. CALENDÁRIO BASEADO NA DATA (RESOLVE O VALUE_ERROR DO RESHAPE)
    # --------------------------------------------------------------------------
    tempos_tr  = sorted(df_tr_sort[VARS['data']].unique())
    tempos_val = sorted(df_val_sort[VARS['data']].unique())
    tempos_tgt = sorted(df_tgt_sort[VARS['data']].unique())

    # Mapeia a semana_epi real correspondente a cada data única para o "Switching" da Loss
    epiweeks_tr  = df_tr_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100
    epiweeks_val = df_val_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100
    epiweeks_tgt = df_tgt_sort.groupby(VARS['data'])[VARS['semana_epi']].first().values % 100

    # Identificação das features de cada regime
    cols_seca = [c for c in COLS_SECA if c in df_tr_sort.columns]
    cols_pico = [c for c in COLS_PICO if c in df_tr_sort.columns]

    num_features_seca = len(cols_seca)
    num_features_pico = len(cols_pico)

    idx_vel_seca  = cols_seca.index('velocidade') if 'velocidade' in cols_seca else 0
    idx_acel_seca = cols_seca.index('aceleracao') if 'aceleracao' in cols_seca else 1
    idx_vel_pico  = cols_pico.index('velocidade') if 'velocidade' in cols_pico else 0
    idx_acel_pico = cols_pico.index('aceleracao') if 'aceleracao' in cols_pico else 1

    # Escalonamento Independente por Regime
    X_tr_sc_seca  = df_tr_sort[cols_seca].fillna(0).values.astype(np.float16)
    X_val_sc_seca = df_val_sort[cols_seca].fillna(0).values.astype(np.float16)
    X_tgt_sc_seca = df_tgt_sort[cols_seca].fillna(0).values.astype(np.float16)

    X_tr_sc_pico  = df_tr_sort[cols_pico].fillna(0).values.astype(np.float16)
    X_val_sc_pico = df_val_sort[cols_pico].fillna(0).values.astype(np.float16)
    X_tgt_sc_pico = df_tgt_sort[cols_pico].fillna(0).values.astype(np.float16)

    # Conversores para GPU
    def _to_tensor(X_sc, tempos, num_f): return torch.tensor(X_sc.reshape(len(tempos), num_nodes, num_f), dtype=torch.float16).to(device)
    def _pop_tensor(df_sort, tempos): return torch.tensor(df_sort['log_pop'].values.reshape(len(tempos), num_nodes), dtype=torch.float16).to(device)
    def _casos_tensor(df_sort, tempos): return torch.tensor(df_sort[VARS['casos_dengue']].fillna(0).values.reshape(len(tempos), num_nodes), dtype=torch.float32).to(device)

    # Criação dos Tensores Finais
    X_tr_t_seca, X_val_t_seca, X_tgt_t_seca = _to_tensor(X_tr_sc_seca, tempos_tr, num_features_seca), _to_tensor(X_val_sc_seca, tempos_val, num_features_seca), _to_tensor(X_tgt_sc_seca, tempos_tgt, num_features_seca)
    X_tr_t_pico, X_val_t_pico, X_tgt_t_pico = _to_tensor(X_tr_sc_pico, tempos_tr, num_features_pico), _to_tensor(X_val_sc_pico, tempos_val, num_features_pico), _to_tensor(X_tgt_sc_pico, tempos_tgt, num_features_pico)

    logpop_tr_t, logpop_val_t, logpop_tgt_t = _pop_tensor(df_tr_sort, tempos_tr), _pop_tensor(df_val_sort, tempos_val), _pop_tensor(df_tgt_sort, tempos_tgt)
    Y_tr_t, Y_val_t = _casos_tensor(df_tr_sort, tempos_tr), _casos_tensor(df_val_sort, tempos_val)
    Y_val_t_np = Y_val_t.cpu().numpy()

    # Limpeza de RAM agressiva
    del df_tr, df_tr_real, df_val_int, df_tr_sort, df_val_sort
    del X_tr_sc_seca, X_val_sc_seca, X_tgt_sc_seca, X_tr_sc_pico, X_val_sc_pico, X_tgt_sc_pico
    import gc
    gc.collect()

    # ==========================================================================
    # SUBFUNÇÃO DE TREINAMENTO DO ESPECIALISTA (Early Stopping via LOSS)
    # ==========================================================================
    def treinar_especialista(num_features, idx_v, idx_a, is_pico_mode, nome, X_tr_t, X_val_t):
        modelo = torch.compile(PreditorNBAGCRN(num_nodes, num_features, 64, idx_v, idx_a).to(device))

        # O modelo de Seca precisa de um weight_decay ligeiramente maior
        w_decay = 1e-4 if is_pico_mode else 1e-3
        otimizador = torch.optim.AdamW(modelo.parameters(), lr=0.004, weight_decay=w_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(otimizador, patience=5, factor=0.5, min_lr=1e-5)

        # 🔥 MUDANÇA: Monitorando a Loss ao invés do MAE
        melhor_loss_val = float('inf')
        sem_melhora = 0
        melhor_estado = None
        melhor_epoca = -1

        # [NOVO] checkpoint por época no Drive: se a sessão cair, rodar tudo de novo retoma daqui
        caminho_ckpt = f"ckpt_{nome}_s{season_id}_v3.pt"   # [V3] nome novo (não reaproveita a v2)
        epoca_ini = 0
        if RETOMAR and os.path.exists(caminho_ckpt):
            ck = torch.load(caminho_ckpt, map_location=device, weights_only=False)
            modelo.load_state_dict(ck['modelo'])
            otimizador.load_state_dict(ck['otimizador'])
            scheduler.load_state_dict(ck['scheduler'])
            melhor_loss_val, sem_melhora, melhor_estado = ck['melhor_loss_val'], ck['sem_melhora'], ck['melhor_estado']
            melhor_epoca = ck.get('melhor_epoca', -1)
            epoca_ini = ck['epoca'] + 1
            if ck['concluido']:
                print(f"   ♻️ {nome}: treino já concluído no checkpoint — usando os pesos salvos")
                modelo.load_state_dict(melhor_estado)
                print(f"      (pesos da época {melhor_epoca + 1}, loss val {melhor_loss_val:.4f})")
                return modelo
            print(f"   ♻️ {nome}: retomando da época {epoca_ini} (melhor loss até aqui: {melhor_loss_val:.4f})")

        def _salvar_ckpt(epoca, concluido):
            torch.save({'modelo': modelo.state_dict(), 'otimizador': otimizador.state_dict(),
                        'scheduler': scheduler.state_dict(), 'melhor_loss_val': melhor_loss_val,
                        'sem_melhora': sem_melhora, 'melhor_estado': melhor_estado, 'melhor_epoca': melhor_epoca,
                        'epoca': epoca, 'concluido': concluido}, caminho_ckpt + '.tmp')
            os.replace(caminho_ckpt + '.tmp', caminho_ckpt)   # troca atômica: nunca fica checkpoint corrompido
            if concluido or (epoca + 1) % 5 == 0:
                salvar_no_drive(caminho_ckpt)                   # cópia no Drive a cada 5 épocas

        barra = tqdm(range(epoca_ini, epochs_max), desc=f' Treino {nome} (S{season_id})', unit='ep', file=sys.stdout,
                     initial=epoca_ini, total=epochs_max)
        epoch = epoca_ini - 1

        for epoch in barra:
            modelo.train()
            h = None

            for t in range(len(tempos_tr) - 1):
                otimizador.zero_grad()

                week_pura = epiweeks_tr[t + 1]
                pertence_ao_pico = (week_pura >= 47) or (week_pura <= 25)

                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    log_mu, log_r, h = modelo(X_tr_t[t], logpop_tr_t[t], h)
                    h = h.detach().clone()

                    # [V3] semana do próprio regime: peso 1; do outro regime: PESO_FORA_REGIME
                    peso = 1.0 if pertence_ao_pico == is_pico_mode else PESO_FORA_REGIME
                    if peso > 0:
                        alvo = Y_tr_t[t + 1].unsqueeze(-1)
                        if is_pico_mode:
                            loss = nb_loss_pico(log_mu, log_r, alvo) * peso
                        else:
                            loss = nb_loss_seca(log_mu, log_r, alvo) * peso

                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                        otimizador.step()

            # --- Validação Estratificada (Calculando LOSS) ---
            modelo.eval()
            loss_val_total = 0.0
            mae_val_total = 0.0
            passos_validos = 0

            with torch.no_grad():
                h_val = None
                with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
                    # Loop ajustado para len - 1, igual ao treino, para prever o tempo t+1
                    for t in range(len(tempos_val) - 1):
                        log_mu_v, log_r_v, h_val = modelo(X_val_t[t], logpop_val_t[t], h_val)
                        if h_val is not None: h_val = h_val.clone()
                        if t + 1 < AQUEC_VAL:      # [V2] semanas de aquecimento: só avançam o estado
                            continue

                        week_val = epiweeks_val[t + 1]
                        val_pertence_pico = (week_val >= 47) or (week_val <= 25)

                        if val_pertence_pico == is_pico_mode:
                            alvo_v = Y_val_t[t + 1].unsqueeze(-1)

                            # 1. Avalia a Loss (NLL + Penalidades)
                            if is_pico_mode:
                                loss_v = nb_loss_pico(log_mu_v, log_r_v, alvo_v)
                            else:
                                loss_v = nb_loss_seca(log_mu_v, log_r_v, alvo_v)

                            loss_val_total += loss_v.item()

                            # 2. Calcula MAE só para visualização no console
                            pred_mu = torch.exp(log_mu_v)
                            mae_v = torch.abs(pred_mu - alvo_v).mean().item()
                            mae_val_total += mae_v

                            passos_validos += 1

            if passos_validos == 0:                    # [V2] nunca mais treina "às cegas"
                raise RuntimeError(f"{nome}: nenhuma semana de validação do regime — aumente SEMANAS_VALIDACAO")
            loss_val_media = loss_val_total / passos_validos
            mae_val_media = mae_val_total / passos_validos

            # Scheduler e Early Stopping dependem exclusivamente da LOSS de validação do regime
            scheduler.step(loss_val_media)

            # [V2] EARLY STOPPING: melhora = queda relativa maior que MIN_DELTA; guarda os pesos da melhor época
            if melhor_estado is None or loss_val_media < melhor_loss_val - abs(melhor_loss_val) * MIN_DELTA:
                melhor_loss_val = loss_val_media
                melhor_estado = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
                melhor_epoca = epoch
                sem_melhora = 0
            else:
                sem_melhora += 1
            barra.set_postfix({'Loss_V': f'{loss_val_media:.3f}', 'MAE_V': f'{mae_val_media:.1f}',
                               'melhor': f'{melhor_loss_val:.3f} (ép {melhor_epoca + 1})',
                               'sem_melhora': f'{sem_melhora}/{patience}'})
            parar = sem_melhora >= patience and epoch + 1 >= EPOCHS_MIN

            if parar or (epoch + 1) % CKPT_A_CADA_EPOCAS == 0:
                _salvar_ckpt(epoch, concluido=False)   # [NOVO]
            if parar:
                print(f"\n   ⏹️ {nome}: early stopping na época {epoch + 1} — {patience} épocas sem melhora; "
                      f"melhor época {melhor_epoca + 1} (loss val {melhor_loss_val:.4f})")
                break

        _salvar_ckpt(epoch, concluido=True)   # [NOVO]
        modelo.load_state_dict(melhor_estado)   # [V2] restaura os pesos da melhor época
        torch.save(melhor_estado, f"modelo_agcrn_{nome}_s{season_id}.pt")
        print(f"   ✅ {nome}: {epoch + 1} épocas rodadas; usando os pesos da época {melhor_epoca + 1} "
              f"(loss val {melhor_loss_val:.4f})")

        return modelo

    # ==========================================================================
    # EXECUÇÃO DO TREINAMENTO E GERAÇÃO DO ALVO RECONCILIADO
    # ==========================================================================
    modelo_seca = treinar_especialista(num_features_seca, idx_vel_seca, idx_acel_seca, is_pico_mode=False, nome="SECA", X_tr_t=X_tr_t_seca, X_val_t=X_val_t_seca)
    modelo_pico = treinar_especialista(num_features_pico, idx_vel_pico, idx_acel_pico, is_pico_mode=True,  nome="PICO", X_tr_t=X_tr_t_pico, X_val_t=X_val_t_pico)

    modelo_seca.eval()
    modelo_pico.eval()
    # [V2] INFERÊNCIA ALINHADA COM O TREINO: no treino a saída de X[t] é comparada com y[t+1];
    # antes, na inferência, essa mesma saída era gravada na linha t (comparada com y[t]).
    # Agora: o estado é aquecido nas semanas EW26–40 (DF_CTX) e a saída da semana t-1 é a previsão da semana t.
    df_ctx_sort = DF_CTX.drop(columns=['node_id', 'time_index'], errors='ignore') \
        .merge(mapa_simples, on=VARS['municipio'], how='left').sort_values([VARS['data'], 'node_id'])
    n_ctx = df_ctx_sort[VARS['data']].nunique()
    Xc_seca = torch.tensor(df_ctx_sort[cols_seca].fillna(0).values.astype(np.float16).reshape(n_ctx, num_nodes, -1)).to(device)
    Xc_pico = torch.tensor(df_ctx_sort[cols_pico].fillna(0).values.astype(np.float16).reshape(n_ctx, num_nodes, -1)).to(device)
    lp_ctx = torch.tensor(df_ctx_sort['log_pop'].values.reshape(n_ctx, num_nodes), dtype=torch.float16).to(device)
    X_all_seca = torch.cat([Xc_seca, X_tgt_t_seca])
    X_all_pico = torch.cat([Xc_pico, X_tgt_t_pico])
    lp_all = torch.cat([lp_ctx, logpop_tgt_t])
    print(f"🔗 Inferência alinhada: {n_ctx} semanas de aquecimento + {len(tempos_tgt)} semanas previstas")

    preds_mu_tgt, preds_r_tgt = [], []
    with torch.no_grad():
        h_seca, h_pico = None, None
        with torch.amp.autocast(device_type='cuda', dtype=torch.float16):
            for t in range(n_ctx + len(tempos_tgt) - 1):
                log_mu_s, log_r_s, h_seca = modelo_seca(X_all_seca[t], lp_all[t], h_seca)
                log_mu_p, log_r_p, h_pico = modelo_pico(X_all_pico[t], lp_all[t], h_pico)
                if h_seca is not None: h_seca = h_seca.clone()
                if h_pico is not None: h_pico = h_pico.clone()

                k = t + 1 - n_ctx                      # semana do teste prevista por esta saída
                if k < 0:
                    continue
                week_pura = epiweeks_tgt[k]            # regime da semana PREVISTA (igual ao treino)
                pertence_ao_pico = (week_pura >= 47) or (week_pura <= 25)
                if pertence_ao_pico:
                    preds_mu_tgt.append(torch.exp(log_mu_p).squeeze(-1))
                    preds_r_tgt.append(torch.exp(log_r_p).squeeze(-1))
                else:
                    preds_mu_tgt.append(torch.exp(log_mu_s).squeeze(-1))
                    preds_r_tgt.append(torch.exp(log_r_s).squeeze(-1))

    assert len(preds_mu_tgt) == len(tempos_tgt)
    preds_mu_tgt_np = torch.stack(preds_mu_tgt).cpu().numpy()
    preds_r_tgt_np = torch.stack(preds_r_tgt).cpu().numpy()

    return {
        'season_id': season_id,
        'preds_mu_np': np.maximum(preds_mu_tgt_np, 0),
        'preds_r_np': np.maximum(preds_r_tgt_np, 1e-3),
        'df_tgt_sort': df_tgt_sort,
        'mapa_simples': mapa_simples,
    }

## 6. Execução principal (célula 48)

In [ ]:
# @title RODAR O MODELO 2
if __name__ == '__main__':
    t_inicio = time.time()
    print(f"\n{'='*75}")
    print(f" 🚀 INICIANDO PIPELINE FINAL (DUPLO ESPECIALISTA): SEASON {SEASON_ALVO}")
    print(f"{'='*75}")

    db_conn = sqlite3.connect(DB_LOCAL)

    # [NOVO] caminho configurável (célula 2)
    # [V2] bancos com as features do SHAP corrigido + semanas de aquecimento
    arquivo_treino = os.path.join(PASTA_DADOS, f"train{SEASON_ALVO}_FINAL_NN_v2.parquet")
    arquivo_teste  = os.path.join(PASTA_DADOS, f"test{SEASON_ALVO}_FINAL_NN_v2.parquet")
    arquivo_ctx    = os.path.join(PASTA_DADOS, f"ctx{SEASON_ALVO}_FINAL_NN_v2.parquet")
    DF_CTX = pd.read_parquet(arquivo_ctx)   # semanas EW26–40: só aquecem o estado da rede na inferência

    print(f"📥 Carregando tensores padronizados:\n   - {arquivo_treino}\n   - {arquivo_teste}\n   - {arquivo_ctx}")
    df_train = pd.read_parquet(arquivo_treino)
    df_test  = pd.read_parquet(arquivo_teste)

    df_train = df_train[df_train[VARS['uf_code']] != 32].reset_index(drop=True)
    df_test  = df_test[df_test[VARS['uf_code']] != 32].reset_index(drop=True)

    n_antes = len(df_train) + len(df_test)
    df_train = df_train.drop_duplicates(subset=[VARS['municipio'], VARS['data']]).reset_index(drop=True)
    df_test  = df_test.drop_duplicates(subset=[VARS['municipio'], VARS['data']]).reset_index(drop=True)
    print(f"🔪 Duplicatas removidas: {n_antes - len(df_train) - len(df_test)} (esperado 0 com os bancos corrigidos)")

    print("🧹 Compactando memória RAM (Downcasting)...")
    for df in [df_train, df_test]:
        colunas_float = df.select_dtypes(include=['float64']).columns
        df[colunas_float] = df[colunas_float].astype('float32')
        colunas_int = df.select_dtypes(include=['int64']).columns
        df[colunas_int] = df[colunas_int].astype('int32')

    mapa_simples = df_train[[VARS['municipio']]].drop_duplicates().sort_values(VARS['municipio']).reset_index(drop=True)
    mapa_simples['node_id'] = mapa_simples.index
    num_nodes = len(mapa_simples)
    print(f"🗺️ Malha Espacial validada: {num_nodes} municípios computáveis.")

    if SO_POS_PROCESSAMENTO:                   # [V3]
        print("⏭️ SO_POS_PROCESSAMENTO = True: pulando o treino e usando os pesos já salvos")
    else:
        res_agcrn = treinar_e_validar_season(
            df_train=df_train,
            df_test=df_test,
            season_id=SEASON_ALVO,
            encoder_means_globais={},
            num_nodes=num_nodes,
            mapa_simples=mapa_simples,
            epochs_max=EPOCHS_MAX,
            patience=PATIENCE
        )
        del res_agcrn
        print(f"⏱️ Treino concluído em {(time.time() - t_inicio) / 60:.1f} min")
    del df_train, df_test
    gc.collect()

    # [V3] previsão + calibração na validação + soma estadual (substitui montar_submissao + MinT)
    df_mun_rec, df_est_rec = prever_e_agregar_calibrado(SEASON_ALVO)

    print("💾 Salvando projeções no banco de dados SQLite...")
    df_mun_rec['season'] = SEASON_ALVO
    df_est_rec['season'] = SEASON_ALVO
    # [NOVO] apaga resultados anteriores DESTA season antes de gravar (antes era 'append' e misturava execuções)
    for tabela in ('municipal_results', 'estadual_results'):
        if db_conn.execute("SELECT name FROM sqlite_master WHERE type='table' AND name=?", (tabela,)).fetchone():
            db_conn.execute(f"DELETE FROM {tabela} WHERE season = ?", (SEASON_ALVO,))
    db_conn.commit()
    df_mun_rec.to_sql('municipal_results', db_conn, if_exists='append', index=False)
    df_est_rec.to_sql('estadual_results', db_conn, if_exists='append', index=False)
    db_conn.close()
    df_est_rec.to_csv(f"previsoes_estaduais_s{SEASON_ALVO}.csv", index=False)   # [NOVO] cópia legível
    for f in os.listdir('.'):                                                      # [NOVO] resultados -> Drive
        if not f.endswith('.parquet') and not f.endswith('.tmp'):
            salvar_no_drive(f)

    print(f"\n📦 Arquivos em {os.getcwd()}:")
    for f in sorted(os.listdir('.')):
        print(f"   - {f}  ({os.path.getsize(f) / 1e6:.1f} MB)")

    del df_mun_rec, df_est_rec
    gc.collect()
    print(f"\n✅ SEASON {SEASON_ALVO} COMPLETA em {(time.time() - t_inicio) / 60:.1f} min.")

## 7. Avaliação (célula 49) e gráfico por UF (célula 50)

In [ ]:
# @title Avaliar Desempenho
import sqlite3
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr

def avaliar_desempenho_season(db_path: str = 'resultados_agcrn.db', season_id: int = 1):
    """
    Conecta ao banco de dados SQLite, extrai as predições reconciliadas (MinT)
    e calcula as métricas de validação científica (MAE, RMSE, sMAPE, Pearson)
    e Métricas Probabilísticas (Cobertura de Intervalos - PIC).
    """
    # 1. Conexão e extração de dados
    if not __import__('os').path.exists(db_path):
        print(f"❌ Banco de dados '{db_path}' não encontrado.")
        return

    conn = sqlite3.connect(db_path)
    df_mun = pd.read_sql(f"SELECT * FROM municipal_results WHERE season = {season_id}", conn)
    df_est = pd.read_sql(f"SELECT * FROM estadual_results WHERE season = {season_id}", conn)
    conn.close()

    if df_mun.empty or df_est.empty:
        print(f"❌ Nenhum registro encontrado para a Season {season_id} no banco '{db_path}'.")
        return

    # 2. Função matemática interna para o bloco de métricas (Incluindo Probabilidade)
    def _calcular_metricas(df):
        y_real = df['casos_reais'].fillna(0.0).values
        y_pred = df['pred'].fillna(0.0).values

        # Erros Absolutos e Quadráticos
        mae = mean_absolute_error(y_real, y_pred)
        rmse = np.sqrt(mean_squared_error(y_real, y_pred))

        # sMAPE protegido
        denominador = (np.abs(y_real) + np.abs(y_pred)) / 2
        smape = np.mean(np.where(denominador == 0, 0, np.abs(y_real - y_pred) / denominador)) * 100

        # Correlação de Pearson
        if np.std(y_real) == 0 or np.std(y_pred) == 0:
            pearson = 0.0
        else:
            pearson, _ = pearsonr(y_real, y_pred)

        # ======================================================================
        # CÁLCULO DE COBERTURA PROBABILÍSTICA (PIC)
        # Verifica se o valor real caiu dentro das bandas de confiança do modelo
        # ======================================================================
        coberturas = {}
        for conf in ['50', '80', '90', '95']:
            col_lower = f'lower_{conf}'
            col_upper = f'upper_{conf}'
            if col_lower in df.columns and col_upper in df.columns:
                dentro_do_intervalo = (df['casos_reais'] >= df[col_lower]) & (df['casos_reais'] <= df[col_upper])
                coberturas[conf] = dentro_do_intervalo.mean() * 100
            else:
                coberturas[conf] = 0.0

        return mae, rmse, smape, pearson, coberturas

    # 4. Execução dos cálculos
    mae_m, rmse_m, smape_m, pearson_m, cob_m = _calcular_metricas(df_mun)
    mae_e, rmse_e, smape_e, pearson_e, cob_e = _calcular_metricas(df_est)

    # 5. Exibição do Relatório com Rigor Acadêmico
    print(f"\n" + "="*70)
    print(f"      RELATÓRIO DE PERFORMANCE EPIDEMIOLÓGICA — SEASON {season_id}")
    print("="*70)

    print(f"📊 NÍVEL MUNICIPAL (Granularidade Espaço-Tempo Local):")
    print(f"  • MAE     : {mae_m:.2f} casos (Magnitude média do erro)")
    print(f"  • RMSE    : {rmse_m:.2f} casos (Sensível a erros em surtos máximos)")
    print(f"  • sMAPE   : {smape_m:.2f}% (Erro percentual relativo)")
    print(f"  • Pearson : {pearson_m:.4f} (Sincronia temporal da curva)")
    print(f"  --- Cobertura Probabilística (Incerteza do Modelo) ---")
    print(f"  • Casos dentro do IC 50% : {cob_m['50']:.1f}%")
    print(f"  • Casos dentro do IC 90% : {cob_m['90']:.1f}%")
    print(f"  • Casos dentro do IC 95% : {cob_m['95']:.1f}% (Meta teórica: ~95%)")

    print("\n" + "-"*70)

    print(f"🏛️ NÍVEL ESTADUAL (soma dos municípios, calibrada na validação):")
    print(f"  • MAE     : {mae_e:.2f} casos")
    print(f"  • RMSE    : {rmse_e:.2f} casos")
    print(f"  • sMAPE   : {smape_e:.2f}%")
    print(f"  • Pearson : {pearson_e:.4f}")
    print(f"  --- Cobertura Probabilística Agregada ---")
    print(f"  • Casos dentro do IC 50% : {cob_e['50']:.1f}%")
    print(f"  • Casos dentro do IC 90% : {cob_e['90']:.1f}%")
    print(f"  • Casos dentro do IC 95% : {cob_e['95']:.1f}%")
    print("="*70 + "\n")

# ==============================================================================
# DISPARO DO SCRIPT DE ANÁLISE
# ==============================================================================
if __name__ == '__main__':
    # Altere o ID da Season conforme desejar avaliar
    avaliar_desempenho_season(db_path=DB_LOCAL, season_id=SEASON_ALVO)

In [ ]:
# @title GRÁFICO DE PREDIÇÕES ESTADUAIS
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def plotar_grid_geofacet_brasil_hd(df_estadual_resultados, season_id, col_uf=VARS['uf_code'], y_max_factor=1.5, y_max_absolute=None):
    """
    Gera o grid geofacet do Brasil com controle customizado do limite do Eixo Y
    para evitar que intervalos de confiança extremos prejudiquem a visualização.
    """
    df_plot = df_estadual_resultados.copy()

    # Garantindo que a data seja reconhecida corretamente
    df_plot['date'] = pd.to_datetime(df_plot['date'])
    df_plot = df_plot.sort_values('date')

    # Matriz do Layout Geográfico do Brasil
    grid_brasil = {
        14: {'uf': 'RR', 'row': 0, 'col': 1}, 16: {'uf': 'AP', 'row': 0, 'col': 4},
        12: {'uf': 'AC', 'row': 1, 'col': 0}, 13: {'uf': 'AM', 'row': 1, 'col': 1},
        11: {'uf': 'RO', 'row': 1, 'col': 2}, 15: {'uf': 'PA', 'row': 1, 'col': 3},
        17: {'uf': 'TO', 'row': 1, 'col': 4}, 21: {'uf': 'MA', 'row': 1, 'col': 5},
        23: {'uf': 'CE', 'row': 1, 'col': 6}, 24: {'uf': 'RN', 'row': 1, 'col': 7},
        51: {'uf': 'MT', 'row': 2, 'col': 2}, 52: {'uf': 'GO', 'row': 2, 'col': 3},
        53: {'uf': 'DF', 'row': 2, 'col': 4}, 22: {'uf': 'PI', 'row': 2, 'col': 5},
        26: {'uf': 'PE', 'row': 2, 'col': 6}, 25: {'uf': 'PB', 'row': 2, 'col': 7},
        50: {'uf': 'MS', 'row': 3, 'col': 2}, 31: {'uf': 'MG', 'row': 3, 'col': 3},
        29: {'uf': 'BA', 'row': 3, 'col': 4}, 27: {'uf': 'AL', 'row': 3, 'col': 5},
        28: {'uf': 'SE', 'row': 3, 'col': 6}, 35: {'uf': 'SP', 'row': 4, 'col': 3},
        33: {'uf': 'RJ', 'row': 4, 'col': 4}, 32: {'uf': 'ES', 'row': 4, 'col': 5},
        41: {'uf': 'PR', 'row': 5, 'col': 2}, 42: {'uf': 'SC', 'row': 5, 'col': 3},
        43: {'uf': 'RS', 'row': 6, 'col': 2}
    }

    fig, axes = plt.subplots(7, 8, figsize=(24, 20))
    for ax in axes.flat: ax.axis('off')

    for code, info in grid_brasil.items():
        ax = axes[info['row'], info['col']]
        df_uf = df_plot[df_plot[col_uf] == code].dropna(subset=['pred', 'date'])

        if df_uf.empty: continue

        ax.axis('on')
        # Plot das bandas de incerteza
        ax.fill_between(df_uf['date'], df_uf['lower_95'], df_uf['upper_95'],
                        color='#1e88e5', alpha=0.15, label='IC 95%' if code==14 else "")

        # Plot da predição reconciliada
        ax.plot(df_uf['date'], df_uf['pred'], color='#00468b', linewidth=1.5,
                label='Predito' if code==14 else "")

        # Plot dos dados reais
        ax.plot(df_uf['date'], df_uf['casos_reais'], color='#d32f2f', linewidth=1.2, linestyle=':',
                label='Real' if code==14 else "")

        ax.set_title(f"📍 {info['uf']}", fontsize=10, weight='bold')
        ax.tick_params(axis='both', which='major', labelsize=7)

        # ----------------------------------------------------------------------
        # CONTROLE DO EIXO Y (Ignora os picos do IC95% na escala do gráfico)
        # ----------------------------------------------------------------------
        if y_max_absolute is not None:
            ax.set_ylim(bottom=0, top=y_max_absolute)
        elif y_max_factor is not None:
            # Encontra o pico real da epidemia no estado (linha central ou casos reais)
            pico_visivel = max(df_uf['pred'].max(), df_uf['casos_reais'].max())
            ax.set_ylim(bottom=0, top=pico_visivel * y_max_factor)
        else:
            ax.set_ylim(bottom=0)

    fig.legend(loc='upper right', bbox_to_anchor=(0.9, 0.95), fontsize=12)
    plt.suptitle(f'🗺️ AGCRN bimodal v3 — previsão estadual calibrada (Season {season_id})', fontsize=20, weight='bold')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    nome_arquivo = f'validacao_brasil_s{season_id}_4K.png'
    plt.savefig(nome_arquivo, dpi=200, bbox_inches='tight')
    plt.show()

    salvar_no_drive(nome_arquivo)   # [NOVO] cópia no Drive, sem download automático
    print(f"🖼️ Gráfico salvo em {os.path.join(PASTA_SAIDA, nome_arquivo)}")

# Execução
import sqlite3
db_conn = sqlite3.connect(DB_LOCAL)

# Carrega a última submissão para a temporada atual
df_est_rec = pd.read_sql_query(f"SELECT * FROM estadual_results WHERE season = {SEASON_ALVO}", db_conn)
db_conn.close()

# Agora sim, chame a função de plotagem
plotar_grid_geofacet_brasil_hd(df_est_rec, season_id=SEASON_ALVO)

## 8. (Opcional) Referência: persistência ingênua

"A semana que vem terá os mesmos casos desta semana", no nível estadual, com os mesmos dados de teste.
Os modelos recebem os casos observados de cada semana do teste, então este é o mínimo que eles precisam superar.
Na avaliação feita no mapa do projeto, deu MAE ≈ 197 e r ≈ 0,98 nos bancos antigos.
A 1ª semana do teste usa a última semana de contexto (EW40), a mesma informação que o modelo tem.

In [ ]:
# @title Persistência ingênua (nível estadual)
tr_ = pd.read_parquet(os.path.join(PASTA_DADOS, f"ctx{SEASON_ALVO}_FINAL_NN_v2.parquet"), columns=['uf_code', 'date', 'casos.x'])  # [V2] EW26–40
te_ = pd.read_parquet(os.path.join(PASTA_DADOS, f"test{SEASON_ALVO}_FINAL_NN_v2.parquet"), columns=['uf_code', 'date', 'casos.x'])
est_te = te_.groupby(['uf_code', 'date'])['casos.x'].sum().reset_index().sort_values(['uf_code', 'date'])
ultimo_tr = tr_.groupby(['uf_code', 'date'])['casos.x'].sum().reset_index().sort_values('date').groupby('uf_code')['casos.x'].last()
est_te['persist'] = est_te.groupby('uf_code')['casos.x'].shift(1)
est_te['persist'] = est_te['persist'].fillna(est_te['uf_code'].map(ultimo_tr))
y, p = est_te['casos.x'].to_numpy(float), est_te['persist'].to_numpy(float)
den = (np.abs(y) + np.abs(p)) / 2
print(f"Persistência estadual S{SEASON_ALVO}: MAE {np.mean(np.abs(y - p)):.2f} · RMSE {np.sqrt(np.mean((y - p) ** 2)):.2f} · "
      f"sMAPE {np.mean(np.where(den == 0, 0, np.abs(y - p) / np.where(den == 0, 1, den))) * 100:.2f}% · r {pearsonr(y, p)[0]:.4f}")
del tr_, te_